# 28: Containment Is Not Individuation

This notebook makes the construct separation executable: raw causal containment can be real while privileged causal individuality is not established once same-checkpoint geometry-matched controls are introduced.

In [1]:
from pathlib import Path
import json
import math
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

NOTEBOOK_PROFILE = os.environ.get("NOTEBOOK_PROFILE", "quick")
RUN_CANONICAL = os.environ.get("RUN_CANONICAL", "0") == "1"


def find_repo_root(start=Path.cwd()):
    current = Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "content" / "books" / "digital-life").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not locate repository root")

REPO_ROOT = find_repo_root()
NOTEBOOK_DIR = REPO_ROOT / "notebooks"
FIG_DIR = NOTEBOOK_DIR / "generated-figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


def load_json(relative_path):
    path = REPO_ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(path)
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def require_path(relative_path):
    path = REPO_ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def summarize_result(label, result, status=None, source="canonical research artifact"):
    row = {"label": label, "source": source}
    if status is not None:
        row["status"] = status
    for key in ["n", "mean", "ci95_low", "ci95_high", "achieved_mde80_one_sided"]:
        if key in result:
            row[key] = result[key]
    return row

print("profile", NOTEBOOK_PROFILE, "run_canonical", RUN_CANONICAL)
print("repo", REPO_ROOT)

CHAPTER = 28
MANUSCRIPT = require_path("content/books/digital-life/28-containment-is-not-individuation/index.md")
LINEAGE = [
    "scripts/books/digital-life/ch28_digital_crystal_causal_modularity_v1.py",
    "scripts/books/digital-life/ch28_digital_crystal_causal_modularity_v2.py",
]
for item in LINEAGE:
    require_path(item)
print("manuscript", MANUSCRIPT.relative_to(REPO_ROOT))
print("lineage ok", len(LINEAGE))

profile quick run_canonical False
repo C:\Projects\working-book
manuscript content\books\digital-life\28-containment-is-not-individuation\index.md
lineage ok 2


## Construct Boundary

V1 tests whether a selected spatial region retains internally initiated causal mass more than external-shell perturbations penetrate it. V2 asks the stronger question: does that selected region exceed same-checkpoint geometry-matched arbitrary controls?

In [2]:
v1_primary = load_json("research/digital-life/ch28-causal-modularity-v1/stage-03-primary.json")
v1_verdict = load_json("research/digital-life/ch28-causal-modularity-v1/stage-06-verdict.json")
v2_primary = load_json("research/digital-life/ch28-causal-modularity-v2/stage-04-primary.json")
v2_verdict = load_json("research/digital-life/ch28-causal-modularity-v2/stage-06-verdict.json")

pd.DataFrame([
    summarize_result("V1 raw module score", v1_primary["module_score"], v1_primary["status"]),
    summarize_result("V2 observed module score", v2_primary["observed_module_score"], "DESCRIPTIVE_COMPONENT"),
    summarize_result("V2 matched-control module score", v2_primary["matched_control_module_score"], "DESCRIPTIVE_COMPONENT"),
    summarize_result("V2 observed minus matched-control excess", v2_primary["result"], v2_primary["status"]),
])

,label,source,status,n,mean,ci95_low,ci95_high,achieved_mde80_one_sided
0,V1 raw module score,canonical research artifact,SUPPORTED,192,0.440209,0.419378,0.461449,0.026781
1,V2 observed module score,canonical research artifact,DESCRIPTIVE_COMPONENT,192,0.443628,0.421161,0.465814,0.028461
2,V2 matched-control module score,canonical research artifact,DESCRIPTIVE_COMPONENT,192,0.455893,0.439013,0.473302,0.021824
3,V2 observed minus matched-control excess,canonical research artifact,BOUNDED_BELOW_SEI,192,-0.012265,-0.032730,0.007197,0.026480


In [3]:
plot_df = pd.DataFrame([
    {"estimand": "raw selected region", "mean": v1_primary["module_score"]["mean"], "lo": v1_primary["module_score"]["ci95_low"], "hi": v1_primary["module_score"]["ci95_high"], "threshold": v1_primary["SEI"]},
    {"estimand": "observed region", "mean": v2_primary["observed_module_score"]["mean"], "lo": v2_primary["observed_module_score"]["ci95_low"], "hi": v2_primary["observed_module_score"]["ci95_high"], "threshold": None},
    {"estimand": "matched control", "mean": v2_primary["matched_control_module_score"]["mean"], "lo": v2_primary["matched_control_module_score"]["ci95_low"], "hi": v2_primary["matched_control_module_score"]["ci95_high"], "threshold": None},
    {"estimand": "excess over control", "mean": v2_primary["result"]["mean"], "lo": v2_primary["result"]["ci95_low"], "hi": v2_primary["result"]["ci95_high"], "threshold": v2_primary["EXCESS_SEI"]},
])
fig, ax = plt.subplots(figsize=(8, 3.6))
y = range(len(plot_df))
ax.errorbar(plot_df["mean"], y, xerr=[plot_df["mean"]-plot_df["lo"], plot_df["hi"]-plot_df["mean"]], fmt="o", color="#3978c5")
ax.axvline(0, color="#333333", lw=1)
ax.axvline(v1_primary["SEI"], color="#888888", lw=1, ls="--", label="raw SEI")
ax.axvline(v2_primary["EXCESS_SEI"], color="#c46a2b", lw=1, ls=":", label="excess SEI")
ax.set_yticks(list(y), plot_df["estimand"])
ax.set_title("CANONICAL ARTIFACT: raw containment vs matched excess")
ax.set_xlabel("module score / excess")
ax.legend(frameon=False, fontsize=8)
path = FIG_DIR / "ch28-raw-vs-excess-modularity.png"
fig.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)
path.relative_to(REPO_ROOT)

WindowsPath('notebooks/generated-figures/ch28-raw-vs-excess-modularity.png')

## Assertion / Identity Checks

The far-field zero assertion is a construction-validity check, not a population discovery.

In [4]:
checks = pd.DataFrame([
    {"check": "V1 far expected effect max abs", "value": v1_verdict["validity"]["far_expected_effect_max_abs"], "status": v1_verdict["validity"]["far_zero_assertion_pass"], "role": "ASSERTION / IDENTITY"},
    {"check": "V2 far expected effect max abs", "value": v2_verdict["validity"]["far_expected_effect_max_abs"], "status": v2_verdict["validity"]["far_zero_assertion_pass"], "role": "ASSERTION / IDENTITY"},
    {"check": "V2 same-checkpoint matching", "value": v2_verdict["validity"]["same_checkpoint_matching"], "status": v2_verdict["validity"]["status"], "role": "CONTROL VALIDITY"},
])
assert checks["status"].all() if checks["status"].dtype == bool else True
checks

,check,value,status,role
0,V1 far expected effect max abs,0.0,True,ASSERTION / IDENTITY
1,V2 far expected effect max abs,0.0,True,ASSERTION / IDENTITY
2,V2 same-checkpoint matching,True,PASS,CONTROL VALIDITY


## Result Ledger

- **V1 survives:** raw causal containment / modularity is supported under the operational radius-4 test.
- **V2 narrows interpretation:** excess over same-checkpoint geometry-matched controls is `BOUNDED_BELOW_SEI`.
- **Not justified:** privileged causal individual, self, organism, autonomy, or life.

`CAUSAL RETENTION != CAUSAL INDIVIDUATION`.